# TrialLens training on Kaggle

Runs the zero-shot baseline and LoRA fine-tuning on a free Kaggle GPU (T4 or P100).

Before running:
1. Settings, Accelerator, choose **GPU T4 x2** or **GPU P100**, and switch **Internet on**.
2. Add-ons, Secrets, add `GITHUB_TOKEN` (a fine-grained read-only token for the TrialLens repo) while the repo is private.

Every cell checks the exit code of its commands, so a failure stops the run instead of carrying on.
Outputs land in `/kaggle/working/TrialLens/results` and `/kaggle/working/TrialLens/checkpoints`; the last cell zips them for download.

In [ ]:
# Clone the repository (token is only needed while the repo is private)
import os
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("GITHUB_TOKEN")
if not os.path.exists("/kaggle/working/TrialLens"):
    !git clone -q https://{token}@github.com/AbdullahRasheed452/TrialLens.git /kaggle/working/TrialLens
%cd /kaggle/working/TrialLens
!git pull -q
!git log --oneline -3

def check():
    assert _exit_code == 0, f"command failed with exit code {_exit_code}"

In [ ]:
# Kaggle ships its own CUDA build of torch, so only the remaining packages are pinned here.
# Kaggle's preinstalled torchao is too old for peft and is not needed, so it is removed.
!pip uninstall -y -q torchao
!pip install -q transformers==5.20.2 peft==0.21.2 accelerate==1.15.0 datasets==5.1.0
check()
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Rebuild the fixed split from the Hugging Face dataset and check it against the committed manifest
!python scripts/make_split.py
check()
!python -m pytest -q
check()

In [ ]:
# Zero-shot baseline (same model before fine-tuning) on the validation split
!python scripts/evaluate.py --split val --name base
check()

In [ ]:
# LoRA fine-tuning; the best epoch by validation macro-F1 is saved as lora-best
!python scripts/train_lora.py --run-name r16 --epochs 4
check()

In [ ]:
# Fine-tuned model on the same validation examples, for the before/after comparison
!python scripts/evaluate.py --split val --name r16 --adapter checkpoints/r16/lora-best
check()

In [ ]:
# Package everything for download
!zip -qr /kaggle/working/triallens_outputs.zip results checkpoints/r16/lora-best checkpoints/r16/train_log.json
check()
!ls -lh /kaggle/working/triallens_outputs.zip